# Remapping Metrics

This notebook quantifies the degree of **place cell remapping** between LM8 and LM8_R45 using the LM8-trained place cell ensemble.

The place cells are activated by observations collected in each environment. The resulting activation matrices form the basis for computing spatial and population-level remapping metrics in subsequent cells.

In [ ]:
%load_ext autoreload
%autoreload 2
%matplotlib inline

import os
import xml.etree.ElementTree as ET
os.chdir('..')

import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches

from realm_tools.experiment_lib.loggers import PovDataset, PlaceCellEnsemble
from realm_tools.place_cell_lib import VisualPlaceCellEnsemble
from realm_tools.simulation_lib.environment_parser import parse_all_walls
from realm_tools.simulation_lib.start_position_generator import _outer_boundary
from realm_tools.image_lib.analysis_plots import _draw_maze, _layout

---
## Configuration

In [ ]:
TRAIN_MAZE           = 'lm8'
TEST_MAZE            = 'lm8_r45'
CENTROID_THRESHOLD   = 0.4    # only activations above this fraction of each cell's
                               # max are used when computing the weighted centroid

PLACE_CELL_PATH = f'data/vpce/place_cells/{TRAIN_MAZE}'
TRAIN_DATA_PATH = f'data/vpce/collect_data/{TRAIN_MAZE}'
TEST_DATA_PATH  = f'data/vpce/collect_data/{TEST_MAZE}'
TRAIN_MAZE_XML  = f'simulation/worlds/environments/vpce/{TRAIN_MAZE}.xml'
BASE_IMG_PATH   = f'analysis/base_figures/{TRAIN_MAZE}.png'
FLIP_BASE_IMG   = True

In [ ]:
pc_model = PlaceCellEnsemble.load(PLACE_CELL_PATH)
ensemble = VisualPlaceCellEnsemble(pc_model.centers, pc_model.radii)

print(f"Place cells : {ensemble.n_cells}")
print(f"Feature dim : {ensemble.feature_dim}")
print(f"Method      : {pc_model.method}")
print(f"Trained on  : {pc_model.maze}")

---
## Compute Activations

The same place cell ensemble is activated by observations from both environments. The resulting matrices `A_lm8` and `A_lm8_r45` have shape `(N_observations, N_cells)` and are the input to all downstream metrics.

In [ ]:
# LM8 — baseline environment
train_ds       = PovDataset.load_dataset(TRAIN_DATA_PATH)
features_lm8   = np.array(train_ds.multimodal_features)
poses_lm8      = np.stack([train_ds.x, train_ds.y, train_ds.theta], axis=1)
A_lm8          = ensemble.activate(features_lm8)

# LM8_R45 — landmarks rotated 45°
test_ds        = PovDataset.load_dataset(TEST_DATA_PATH)
features_r45   = np.array(test_ds.multimodal_features)
poses_r45      = np.stack([test_ds.x, test_ds.y, test_ds.theta], axis=1)
A_r45          = ensemble.activate(features_r45)

print(f"A_lm8  : {A_lm8.shape}  —  {TRAIN_MAZE.upper()}")
print(f"A_r45  : {A_r45.shape}  —  {TEST_MAZE.upper()}")

---
## Peak Activation Location

For each place cell the **activation-weighted centroid** of all observation positions gives the spatial location where that cell's activation is concentrated:

$$\bar{x}_i = \frac{\sum_n a_i(n)\, x_n}{\sum_n a_i(n)}, \qquad \bar{y}_i = \frac{\sum_n a_i(n)\, y_n}{\sum_n a_i(n)}$$

This is more robust than argmax — it reflects the full distribution of activation across the maze rather than the single highest-activation observation.

In [ ]:
def weighted_centroids(activations, poses, threshold=0.0):
    """
    Compute the activation-weighted centroid (x, y) for each place cell.

    Activations below `threshold * cell_max` are zeroed before computing
    the weighted mean, ignoring weakly activated locations.

    Parameters
    ----------
    activations : np.ndarray, shape (N, K)
    poses       : np.ndarray, shape (N, 3)  — columns: x, y, theta
    threshold   : float  — fraction of each cell's max activation below which
                           observations are ignored (default 0 = use all)

    Returns
    -------
    np.ndarray, shape (K, 2)  — (x, y) centroid per cell
    """
    xy      = poses[:, :2]
    weights = activations.copy()

    if threshold > 0:
        cell_max = weights.max(axis=0, keepdims=True)   # (1, K)
        weights[weights < threshold * cell_max] = 0

    totals = weights.sum(axis=0, keepdims=True).T        # (K, 1)
    return (weights.T @ xy) / totals                     # (K, 2)


centroids_lm8 = weighted_centroids(A_lm8, poses_lm8, threshold=CENTROID_THRESHOLD)
centroids_r45 = weighted_centroids(A_r45, poses_r45, threshold=CENTROID_THRESHOLD)

print(f"Threshold        : {CENTROID_THRESHOLD:.0%} of each cell's max activation")
print(f"Centroids LM8    : {centroids_lm8.shape}")
print(f"Centroids R45    : {centroids_r45.shape}")

---
## Centroid Shift Map

Each place cell is represented by two dots superimposed on the LM8 base figure:

- 🟢 **Green** — activation-weighted centroid in LM8 (baseline)
- 🔴 **Red** — activation-weighted centroid in LM8_R45 (rotated landmarks)

A line connects each pair to make the shift direction visible. Cells whose centroids shift substantially between environments are remapping in response to the landmark rotation.

In [ ]:
# --- Base image extent from maze XML ---
walls    = parse_all_walls(ET.parse(TRAIN_MAZE_XML).getroot())
boundary = _outer_boundary(walls)
bminx, bminy, bmaxx, bmaxy = boundary.bounds
base_extent = [bminx, bmaxx, bminy, bmaxy]

base_img = plt.imread(BASE_IMG_PATH)
if FLIP_BASE_IMG:
    base_img = np.flipud(base_img)

# --- Layout (same physical sizing as plot_place_cell_activations_overlay) ---
n_cells = len(centroids_lm8)
ncols   = int(np.ceil(np.sqrt(n_cells)))
nrows   = int(np.ceil(n_cells / ncols))

fig_w, fig_h, subplot_top, cbar_bot, cbar_h_n, title_y = _layout(nrows, ncols)

fig, axes = plt.subplots(nrows, ncols,
                          figsize=(fig_w, fig_h),
                          facecolor='white')
axes = axes.flatten()

for i in range(n_cells):
    ax = axes[i]
    ax.set_facecolor('white')

    ax.imshow(base_img, extent=base_extent, origin='lower', aspect='equal', zorder=0)
    _draw_maze(ax, TRAIN_MAZE_XML)

    # Line connecting the two centroids
    ax.plot([centroids_lm8[i, 0], centroids_r45[i, 0]],
            [centroids_lm8[i, 1], centroids_r45[i, 1]],
            color='grey', linewidth=1.2, alpha=0.6, zorder=1)

    # LM8 centroid — green
    ax.scatter(centroids_lm8[i, 0], centroids_lm8[i, 1],
               c='green', s=60, zorder=3, edgecolors='darkgreen', linewidths=0.6)

    # LM8_R45 centroid — red
    ax.scatter(centroids_r45[i, 0], centroids_r45[i, 1],
               c='red', s=60, zorder=3, edgecolors='darkred', linewidths=0.6)

    ax.set_xlim(base_extent[0], base_extent[1])
    ax.set_ylim(base_extent[2], base_extent[3])
    ax.set_title(f'Cell {i}', fontsize=39, color='black', pad=4)
    ax.set_xticks([])
    ax.set_yticks([])
    ax.set_aspect('equal')

for j in range(n_cells, len(axes)):
    axes[j].set_visible(False)

plt.subplots_adjust(top=subplot_top, bottom=0.01, hspace=0.35, wspace=0.2)

fig.suptitle(
    f'Place Cell Centroid Shift — {TRAIN_MAZE.upper()} vs {TEST_MAZE.upper()}\n'
    f'{pc_model.method.upper()}  K={n_cells}',
    fontsize=54, color='black', y=title_y
)

# Shared legend in the header area
legend_ax = fig.add_axes([0.15, cbar_bot, 0.7, cbar_h_n * 3])
legend_ax.axis('off')
legend_ax.legend(
    handles=[
        mpatches.Patch(color='green',  label=f'{TRAIN_MAZE.upper()} — baseline'),
        mpatches.Patch(color='red',    label=f'{TEST_MAZE.upper()} — landmarks +45°'),
    ],
    loc='center', ncol=2, fontsize=36, framealpha=0.0
)

plt.show()